In [2]:
import pandas as pd

print("Jupyter fonctionne correctement !")


Jupyter fonctionne correctement !


In [3]:
from pathlib import Path
import pandas as pd

# Localiser le dossier data
project_dir = Path.cwd()
if not (project_dir / "data").is_dir():
    project_dir = project_dir.parent

# Charger les commandes Olist
orders = pd.read_csv(
    project_dir / "data" / "olist_orders_dataset.csv"
)

print("Dimensions :", orders.shape)
display(orders.head())

Dimensions : (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [5]:
missing = pd.DataFrame({
    "missing_count": orders.isna().sum(),
    "missing_percent": (orders.isna().mean() * 100).round(2)
})

display(missing)

,missing_count,missing_percent
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [6]:
# Commandes sans date de livraison
missing_delivery = orders[
    orders["order_delivered_customer_date"].isna()
]

# Répartition selon le statut
display(
    missing_delivery["order_status"].value_counts()
)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [7]:
# Commandes marquées livrées, mais sans date de livraison
anomalies = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].isna())
]

print("Commandes potentiellement incohérentes :", len(anomalies))
display(anomalies.head())

Commandes potentiellement incohérentes : 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


In [8]:
# Travailler sur une copie pour préserver les données brutes
orders_clean = orders.copy()

# Identifier les commandes livrées sans date de livraison
missing_delivered_date = (
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_customer_date"].isna())
)

# Ajouter un indicateur de qualité
orders_clean["quality_status"] = "valid"

orders_clean.loc[
    missing_delivered_date,
    "quality_status"
] = "review"

# Résultat
display(orders_clean["quality_status"].value_counts())

quality_status
valid     99433
review        8
Name: count, dtype: int64

In [9]:
display(
    orders_clean.loc[
        orders_clean["quality_status"] == "review",
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ]
)

,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-30 18:12:23,NaN
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-25 08:05:00,NaN
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-03 13:57:00,NaN
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-03 13:57:00,NaN
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-03 09:28:00,NaN
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaN,NaN
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-12 14:10:00,NaN
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-07-03 19:26:00,NaN


In [10]:
# Identifier les commandes livrées sans date de livraison
mask = (
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_customer_date"].isna())
)

# Ajouter une colonne décrivant l'anomalie
orders_clean["quality_issue"] = pd.NA

orders_clean.loc[
    mask, "quality_issue"
] = "missing_delivery_date"

# Vérifier
display(
    orders_clean.loc[
        mask,
        ["order_id", "order_status",
         "quality_status", "quality_issue"]
    ]
)

,order_id,order_status,quality_status,quality_issue
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,review,missing_delivery_date
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,review,missing_delivery_date
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,review,missing_delivery_date
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,review,missing_delivery_date
82868,0d3268bad9b086af767785e3f0fc0133,delivered,review,missing_delivery_date
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,review,missing_delivery_date
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,review,missing_delivery_date
98038,20edc82cf5400ce95e1afacc25798b31,delivered,review,missing_delivery_date


In [11]:
# Commandes livrées avec une date de livraison connue
valid_deliveries = orders_clean[
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_customer_date"].notna())
].copy()

# Convertir les dates si nécessaire
valid_deliveries["order_purchase_timestamp"] = pd.to_datetime(
    valid_deliveries["order_purchase_timestamp"]
)

valid_deliveries["order_delivered_customer_date"] = pd.to_datetime(
    valid_deliveries["order_delivered_customer_date"]
)

# Calculer le délai de livraison
valid_deliveries["delivery_days"] = (
    valid_deliveries["order_delivered_customer_date"]
    - valid_deliveries["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Écarter les délais négatifs incohérents
valid_deliveries = valid_deliveries[
    valid_deliveries["delivery_days"] >= 0
]

print(
    "Délai moyen de livraison :",
    round(valid_deliveries["delivery_days"].mean(), 2),
    "jours"
)

Délai moyen de livraison : 12.56 jours


In [12]:
# Sélectionner les commandes sans date d'approbation
missing_approval = orders[
    orders["order_approved_at"].isna()
]

# Compter les commandes selon leur statut
display(
    missing_approval["order_status"].value_counts()
)

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [13]:
# Commandes livrées sans date d'approbation
mask_approval = (
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_approved_at"].isna())
)

# Signaler ces commandes
orders_clean.loc[mask_approval, "quality_status"] = "review"

# Afficher les commandes concernées
display(
    orders_clean.loc[
        mask_approval,
        [
            "order_id",
            "order_status",
            "order_approved_at",
            "order_delivered_customer_date",
            "quality_status"
        ]
    ]
)

,order_id,order_status,order_approved_at,order_delivered_customer_date,quality_status
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,NaN,2017-03-01 13:25:33,review
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,NaN,2017-03-02 10:05:06,review
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,NaN,2017-03-01 08:07:38,review
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,NaN,2017-03-09 07:28:47,review
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,NaN,2017-03-02 11:09:19,review
26800,c1d4211b3dae76144deccd6c74144a88,delivered,NaN,2017-01-30 18:16:01,review
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,NaN,2017-03-02 03:41:58,review
39334,d77031d6a3c8a52f019764e68f211c69,delivered,NaN,2017-03-02 16:15:23,review
48401,7002a78c79c519ac54022d4f8a65e6e8,delivered,NaN,2017-02-06 14:22:19,review
61743,2eecb0d85f281280f79fa00f9cec1a95,delivered,NaN,2017-03-03 12:16:03,review


In [14]:
mask_delivery = (
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_customer_date"].isna())
)

print("Sans date de livraison :", mask_delivery.sum())
print("Sans date d'approbation :", mask_approval.sum())
print("Avec les deux anomalies :", (mask_delivery & mask_approval).sum())

print("\nRésumé de qualité :")
display(orders_clean["quality_status"].value_counts())

Sans date de livraison : 8
Sans date d'approbation : 14
Avec les deux anomalies : 0

Résumé de qualité :


quality_status
valid     99419
review       22
Name: count, dtype: int64

In [15]:
# Convertir les colonnes de dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce"
    )

# 1. Approbation avant achat
invalid_approval = (
    orders_clean["order_approved_at"]
    < orders_clean["order_purchase_timestamp"]
)

# 2. Expédition avant achat
invalid_shipping = (
    orders_clean["order_delivered_carrier_date"]
    < orders_clean["order_purchase_timestamp"]
)

# 3. Livraison avant expédition
invalid_delivery = (
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_delivered_carrier_date"]
)

print("Approbations avant achat :", invalid_approval.sum())
print("Expéditions avant achat :", invalid_shipping.sum())
print("Livraisons avant expédition :", invalid_delivery.sum())

Approbations avant achat : 0
Expéditions avant achat : 166
Livraisons avant expédition : 23


In [16]:
shipping_anomalies = orders_clean.loc[invalid_shipping].copy()

shipping_anomalies["difference_hours"] = (
    shipping_anomalies["order_purchase_timestamp"]
    - shipping_anomalies["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

display(
    shipping_anomalies[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "difference_hours"
        ]
    ].sort_values("difference_hours", ascending=False).head(10)
)

display(shipping_anomalies["difference_hours"].describe())

,order_id,order_purchase_timestamp,order_delivered_carrier_date,difference_hours
25883,7c48bb55e8e4f7e56d412e9653db37bc,2018-07-16 18:40:53,2018-01-26 13:35:00,4109.098056
83321,4021cd7611d6d9ce5ffcd24817fc374f,2018-08-18 11:49:40,2018-08-14 06:22:00,101.461111
67844,db090a16182b263b1e896bb26c6f66cf,2018-07-13 16:14:08,2018-07-13 13:59:00,2.252222
65452,9711d975b961355b4b5d636857e48498,2018-06-13 15:23:50,2018-06-13 13:15:00,2.147222
79401,89d32b64af005178b318f76cd60f2c3c,2018-07-06 11:54:40,2018-07-06 09:48:00,2.111111
13969,6192897f85cb2aff6dd1fca56fddb45e,2018-07-18 13:34:22,2018-07-18 11:33:00,2.022778
43657,3ed559dec69ab96d9074fedaf18650eb,2018-06-15 13:20:09,2018-06-15 11:22:00,1.969167
4256,4e157a36ea9cf89bde6fff57a780b525,2018-08-24 14:37:50,2018-08-24 12:43:00,1.913889
74503,119060af27b04f5fae8b2b0e27eef7f9,2018-06-11 14:34:47,2018-06-11 12:45:00,1.829722
21910,e23c9dcc304042fa90d538be679fa68d,2018-06-13 11:41:23,2018-06-13 09:53:00,1.806389


count     166.000000
mean       26.039478
std       318.923845
min         0.006667
25%         0.234444
50%         0.599861
75%         1.017500
max      4109.098056
Name: difference_hours, dtype: float64

In [17]:
delivery_anomalies = orders_clean.loc[invalid_delivery].copy()

delivery_anomalies["difference_hours"] = (
    delivery_anomalies["order_delivered_carrier_date"]
    - delivery_anomalies["order_delivered_customer_date"]
).dt.total_seconds() / 3600

display(
    delivery_anomalies[
        [
            "order_id",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "difference_hours"
        ]
    ].sort_values("difference_hours", ascending=False).head(10)
)

display(delivery_anomalies["difference_hours"].describe())

,order_id,order_delivered_carrier_date,order_delivered_customer_date,difference_hours
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,2017-03-02 17:34:26,2017-02-14 15:15:57,386.308056
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-09 18:18:43,2017-08-01 21:13:01,189.095000
45302,29941903985f944b0ffc49c479c1547d,2017-06-09 15:07:29,2017-06-02 11:09:16,171.970278
14474,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10,144.238889
49933,76458889992169d3135b264dc13aec67,2016-10-26 11:43:06,2016-10-20 18:03:17,137.663611
71227,19feb5627c41ea1b36a8e50a469b3644,2016-10-26 11:42:05,2016-10-20 19:07:54,136.569722
74967,d5558a097766363b8e76b38c43332e8a,2017-02-15 08:55:26,2017-02-10 07:58:32,120.948333
41636,b866af202be0692766081310cd4085e1,2017-02-20 02:32:08,2017-02-15 03:53:46,118.639444
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56,69.417222
78556,ea1dcb4757a844d2642547797bd5feb0,2017-07-27 19:21:31,2017-07-25 19:43:10,47.639167


count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: difference_hours, dtype: float64

In [18]:
# Règle 1 : livrée sans date de livraison
r1 = (
    (orders_clean["order_status"] == "delivered")
    & orders_clean["order_delivered_customer_date"].isna()
)

# Règle 2 : livrée sans date d'approbation
r2 = (
    (orders_clean["order_status"] == "delivered")
    & orders_clean["order_approved_at"].isna()
)

# Règle 3 : expédition avant achat
r3 = (
    orders_clean["order_delivered_carrier_date"]
    < orders_clean["order_purchase_timestamp"]
)

# Règle 4 : livraison avant expédition
r4 = (
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_delivered_carrier_date"]
)

# Une commande est à examiner si au moins une règle est violée
has_issue = r1 | r2 | r3 | r4

orders_clean["quality_status"] = "valid"
orders_clean.loc[has_issue, "quality_status"] = "review"

print("Commandes à examiner :", has_issue.sum())
print("Commandes sans anomalie détectée :", (~has_issue).sum())

Commandes à examiner : 211
Commandes sans anomalie détectée : 99230


In [19]:
quality_checks = pd.DataFrame({
    "missing_delivery_date": r1,
    "missing_approval_date": r2,
    "shipping_before_purchase": r3,
    "delivery_before_shipping": r4
})

print("Nombre d'anomalies par règle :")
display(quality_checks.sum())

print(
    "Commandes avec plusieurs anomalies :",
    (quality_checks.sum(axis=1) > 1).sum()
)

print("\nRépartition finale :")
display(orders_clean["quality_status"].value_counts())

Nombre d'anomalies par règle :


missing_delivery_date         8
missing_approval_date        14
shipping_before_purchase    166
delivery_before_shipping     23
dtype: int64

Commandes avec plusieurs anomalies : 0

Répartition finale :


quality_status
valid     99230
review      211
Name: count, dtype: int64

In [20]:
# Commandes sans date de remise au transporteur
missing_carrier = orders_clean[
    orders_clean["order_delivered_carrier_date"].isna()
]

# Répartition par statut
display(
    missing_carrier["order_status"].value_counts()
)

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [21]:
# Règle 5 : commande livrée sans date de remise au transporteur
r5 = (
    (orders_clean["order_status"] == "delivered")
    & orders_clean["order_delivered_carrier_date"].isna()
)

print("Anomalies détectées :", r5.sum())

Anomalies détectées : 2


In [22]:
# Les quatre règles précédentes
previous_issues = r1 | r2 | r3 | r4

print(
    "Déjà signalées :",
    (r5 & previous_issues).sum()
)

print(
    "Nouvelles commandes à examiner :",
    (r5 & ~previous_issues).sum()
)

Déjà signalées : 1
Nouvelles commandes à examiner : 1


In [23]:
# Regrouper les cinq règles
all_issues = r1 | r2 | r3 | r4 | r5

orders_clean["quality_status"] = "valid"
orders_clean.loc[all_issues, "quality_status"] = "review"

display(orders_clean["quality_status"].value_counts())

quality_status
valid     99229
review      212
Name: count, dtype: int64

Vérifier les identifiants manquants


In [24]:
print("order_id manquants :", orders_clean["order_id"].isna().sum())
print("customer_id manquants :", orders_clean["customer_id"].isna().sum())

print("order_id vides :", orders_clean["order_id"].str.strip().eq("").sum())
print("customer_id vides :", orders_clean["customer_id"].str.strip().eq("").sum())

order_id manquants : 0
customer_id manquants : 0
order_id vides : 0
customer_id vides : 0


In [25]:
print(
    "Lignes entièrement dupliquées :",
    orders_clean.duplicated().sum()
)

print(
    "order_id dupliqués :",
    orders_clean["order_id"].duplicated().sum()
)

print(
    "customer_id dupliqués :",
    orders_clean["customer_id"].duplicated().sum()
)

Lignes entièrement dupliquées : 0
order_id dupliqués : 0
customer_id dupliqués : 0


In [26]:
customers = pd.read_csv(
    project_dir / "data" / "olist_customers_dataset.csv"
)

# Vérifier que chaque customer_id des commandes
# existe dans la table customers
unknown_customers = orders_clean[
    ~orders_clean["customer_id"].isin(customers["customer_id"])
]

print("Clients introuvables :", len(unknown_customers))

# Vérifier l'unicité des identifiants dans customers
print(
    "customer_id dupliqués dans customers :",
    customers["customer_id"].duplicated().sum()
)

Clients introuvables : 0
customer_id dupliqués dans customers : 0


In [27]:
# Regrouper les règles et leurs noms
rules = {
    "missing_delivery_date": r1,
    "missing_approval_date": r2,
    "shipping_before_purchase": r3,
    "delivery_before_shipping": r4,
    "missing_carrier_date": r5
}

# Ajouter la liste des anomalies pour chaque commande
orders_clean["quality_issues"] = [
    [name for name, mask in rules.items() if mask.loc[idx]]
    for idx in orders_clean.index
]

# Afficher les commandes à examiner
display(
    orders_clean.loc[
        orders_clean["quality_status"] == "review",
        ["order_id", "quality_status", "quality_issues"]
    ].head(15)
)

,order_id,quality_status,quality_issues
615,b9afddbdcfadc9a87b41a83271c3e888,review,[shipping_before_purchase]
1111,ad133696906f6a78826daa0911b7daec,review,[shipping_before_purchase]
1329,74e033208dc13a7b8127eb8e73d09b76,review,[shipping_before_purchase]
1372,a6b58794fd2ba533359a76c08df576e3,review,[shipping_before_purchase]
1864,5792e0b1c8c8a2bf53af468c9a422c88,review,[shipping_before_purchase]
2760,c3eb293fd154223498b6551a728203e8,review,[shipping_before_purchase]
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,review,[missing_delivery_date]
3473,b0c2a7d04b165525254254a728c50a4e,review,[shipping_before_purchase]
3661,2033a4586b5bec3229ebc1675a8ae092,review,[shipping_before_purchase]
4114,08adcddad19d3acf37d1fa01cb9ded1e,review,[shipping_before_purchase]


In [28]:
# Statuts attendus dans Olist
valid_statuses = {
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "canceled",
    "unavailable"
}

# Vérifier les statuts inconnus
invalid_status = (
    ~orders_clean["order_status"].isin(valid_statuses)
)

print("Statuts inconnus :", invalid_status.sum())

# Afficher tous les statuts présents
display(orders_clean["order_status"].value_counts())

Statuts inconnus : 0


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [29]:
# Colonnes contenant des dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Comparer les dates originales aux dates converties
for col in date_columns:
    original = orders[col]

    converted = pd.to_datetime(original, errors="coerce")

    invalid_dates = (
        original.notna()
        & converted.isna()
    )

    print(col, ":", invalid_dates.sum(), "dates invalides")

order_purchase_timestamp : 0 dates invalides
order_approved_at : 0 dates invalides
order_delivered_carrier_date : 0 dates invalides
order_delivered_customer_date : 0 dates invalides
order_estimated_delivery_date : 0 dates invalides


In [30]:
# Règle 6 : livraison avant achat
r6 = (
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_purchase_timestamp"]
)

# Règle 7 : livraison estimée avant achat
r7 = (
    orders_clean["order_estimated_delivery_date"]
    < orders_clean["order_purchase_timestamp"]
)

# Règle 8 : livraison avant approbation
r8 = (
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_approved_at"]
)

print("Livraisons avant achat :", r6.sum())
print("Dates estimées avant achat :", r7.sum())
print("Livraisons avant approbation :", r8.sum())

Livraisons avant achat : 0
Dates estimées avant achat : 0
Livraisons avant approbation : 61


In [31]:
# Regrouper les règles
all_rules = {
    "missing_delivery_date": r1,
    "missing_approval_date": r2,
    "shipping_before_purchase": r3,
    "delivery_before_shipping": r4,
    "missing_carrier_date": r5,
    "delivery_before_purchase": r6,
    "estimated_delivery_before_purchase": r7,
    "delivery_before_approval": r8,
    "invalid_order_status": invalid_status
}

# Matrice des anomalies
quality_checks = pd.DataFrame(all_rules, index=orders_clean.index)

# Nombre de règles violées par commande
orders_clean["quality_issue_count"] = quality_checks.sum(axis=1)

# Statut global
orders_clean["quality_status"] = "valid"
orders_clean.loc[
    orders_clean["quality_issue_count"] > 0,
    "quality_status"
] = "review"

print("Anomalies par règle :")
display(quality_checks.sum())

print("\nCommandes à examiner :")
display(orders_clean["quality_status"].value_counts())

Anomalies par règle :


missing_delivery_date                   8
missing_approval_date                  14
shipping_before_purchase              166
delivery_before_shipping               23
missing_carrier_date                    2
delivery_before_purchase                0
estimated_delivery_before_purchase      0
delivery_before_approval               61
invalid_order_status                    0
dtype: int64


Commandes à examiner :


quality_status
valid     99168
review      273
Name: count, dtype: int64

In [32]:
approval_delivery_anomalies = orders_clean.loc[r8].copy()

approval_delivery_anomalies["difference_hours"] = (
    approval_delivery_anomalies["order_approved_at"]
    - approval_delivery_anomalies["order_delivered_customer_date"]
).dt.total_seconds() / 3600

display(
    approval_delivery_anomalies[
        [
            "order_id",
            "order_approved_at",
            "order_delivered_customer_date",
            "difference_hours"
        ]
    ].sort_values("difference_hours", ascending=False).head(10)
)

display(approval_delivery_anomalies["difference_hours"].describe())

,order_id,order_approved_at,order_delivered_customer_date,difference_hours
31211,bc4854efd86d9f42140c951c595d20c1,2017-09-13 22:00:51,2017-09-06 22:15:45,167.751667
14562,1fab4ac9d85079b3da72a11475ae1685,2017-09-13 22:06:11,2017-09-08 20:13:03,121.885556
41592,8554cb37f7158cb0b082a841d24a4589,2017-09-13 21:58:04,2017-09-08 20:07:45,121.838611
85393,40de47dfa620d667117e4a6067b6e1ec,2017-09-13 21:58:38,2017-09-08 20:41:16,121.289444
91801,4387477eec4b3c89b39f3f454940d059,2018-08-20 15:56:29,2018-08-15 14:56:43,120.996111
50623,e73fe43cdcd166f7f0c6e3c2bf11a917,2018-08-20 15:57:28,2018-08-15 15:46:38,120.180556
61544,6dcf0aeb8b1eb4021c26e1d0e9394979,2018-08-20 15:59:18,2018-08-15 19:06:29,116.880278
25411,6b80bb20190715d71c43efff617bd659,2017-03-01 10:51:46,2017-02-24 14:27:26,116.405556
46163,0184d4ddb259e1a4cfc2871888cf97b8,2017-09-13 22:17:15,2017-09-09 15:12:44,103.075278
87076,bc1b85147b5edbb7cbefcf5c1bd5ded9,2018-07-29 23:30:54,2018-07-25 20:56:35,98.571944


count     61.000000
mean      39.054408
std       42.976017
min        0.088611
25%        3.452222
50%       24.823333
75%       55.826944
max      167.751667
Name: difference_hours, dtype: float64